<a href="https://colab.research.google.com/github/TinaawhyteD/Northstar-RAG-Assistant/blob/main/Northstar_RAG_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers chromadb pypdf transformers accelerate


KeyboardInterrupt: 

In [ ]:
import sentence_transformers
import chromadb
import pypdf
import transformers

print("Sentence Transformers: Ready")
print("ChromaDB: Ready")
print("PyPDF: Ready")
print("Transformers: Ready")

Sentence Transformers: Ready
ChromaDB: Ready
PyPDF: Ready
Transformers: Ready


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "I love building AI agents",
    "I enjoy developing intelligent automation systems",
    "I like eating rice"
]

embeddings = model.encode(sentences)

print("Embedding shape:", embeddings.shape)
print("First embedding:", embeddings[0][:10])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (3, 384)
First embedding: [-0.06462169 -0.09009665 -0.03620673 -0.06142319  0.04242132 -0.03892067
  0.02637081  0.01054153  0.06309438  0.03379731]


In [ ]:
from sentence_transformers import util

similarity_1 = util.cos_sim(embeddings[0], embeddings[1])
similarity_2 = util.cos_sim(embeddings[0], embeddings[2])

print("AI sentences similarity:", similarity_1.item())
print("AI and rice similarity:", similarity_2.item())

AI sentences similarity: 0.5626033544540405
AI and rice similarity: 0.18852144479751587


In [ ]:
from pypdf import PdfReader

pdf_path = "/content/sample_company_handbook_rag_training.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print("Total pages:", len(reader.pages))
print("Total characters:", len(text))

print("\nFIRST 2000 CHARACTERS:\n")
print(text[:2000])

Total pages: 3
Total characters: 6648

FIRST 2000 CHARACTERS:

Northstar Digital | Synthetic RAG training handbook
Page 1
 NORTHSTAR DIGITAL
 EMPLOYEE HANDBOOK & WORKPLACE POLICIES
Synthetic training document — Northstar Digital is a fictional company. All policies, dates, and contact details
in this handbook are invented for learning and RAG practice. They are not real legal or employment advice.
1. Company overview
Northstar Digital is a fictional software and digital services company founded in 2021. Its main office is listed as
14 Meridian Way, Bristol, United Kingdom. The company provides workflow automation, data services, and
customer support software to small and medium-sized businesses.
The company’s stated mission is to make everyday business processes simpler, safer, and easier to measure.
Its fictional workforce includes engineering, customer success, operations, sales, and design teams.
2. Working hours and attendance
Standard working hours are Monday to Friday, 9:00 a.m. 

In [ ]:
# Step 5: Split document into chunks

chunk_size = 800
chunk_overlap = 150

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size

    chunk = text[start:end]

    chunks.append(chunk)

    start += chunk_size - chunk_overlap

print("Total chunks created:", len(chunks))

print("\nFIRST CHUNK:\n")
print(chunks[0])

Total chunks created: 11

FIRST CHUNK:

Northstar Digital | Synthetic RAG training handbook
Page 1
 NORTHSTAR DIGITAL
 EMPLOYEE HANDBOOK & WORKPLACE POLICIES
Synthetic training document — Northstar Digital is a fictional company. All policies, dates, and contact details
in this handbook are invented for learning and RAG practice. They are not real legal or employment advice.
1. Company overview
Northstar Digital is a fictional software and digital services company founded in 2021. Its main office is listed as
14 Meridian Way, Bristol, United Kingdom. The company provides workflow automation, data services, and
customer support software to small and medium-sized businesses.
The company’s stated mission is to make everyday business processes simpler, safer, and easier to measure.
Its fictional workforce includes engineering, custo


In [ ]:
import chromadb

# Create a ChromaDB client
client = chromadb.Client()

# Create a collection
collection = client.create_collection(
    name="northstar_handbook"
)

# Generate embeddings for all chunks
chunk_embeddings = model.encode(chunks).tolist()

# Store chunks and embeddings
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings
)

print("Total documents stored:", collection.count())

Total documents stored: 11


In [ ]:
# Step 7: Retrieve relevant information

question = "How many days of annual leave do employees receive?"

# Convert question into an embedding
question_embedding = model.encode(question).tolist()

# Search ChromaDB
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

# Display retrieved information
print("USER QUESTION:", question)

print("\nRETRIEVED DOCUMENTS:\n")

for i, document in enumerate(results["documents"][0]):
    print(f"Result {i+1}:")
    print(document)
    print("-" * 50)

USER QUESTION: How many days of annual leave do employees receive?

RETRIEVED DOCUMENTS:

Result 1:
ssible if they will be late or absent.
Attendance is recorded through the company’s fictional WorkTrack portal. Employees who cannot access the
portal should email people@northstar.example.
3. Annual leave and other time off
Full-time employees receive 25 days of paid annual leave per calendar year, excluding public holidays observed
in their agreed work location. Part-time leave is calculated proportionally.
Leave requests should be submitted through WorkTrack at least 10 working days before the first requested day.
A manager normally responds within three working days. Approval depends on team coverage and business
needs; employees should not book non-refundable travel before approval.
Employees may take up to five paid sick days per calendar year under this fictional policy. For an abse
--------------------------------------------------
Result 2:
ot book non-refundable travel before a

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

print("Language model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: 'The task text2text-generation has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'

In [1]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

print("Language model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BailingHybridForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BeamForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCa

Language model loaded successfully!


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

# Load language model
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("FLAN-T5 is ready!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 is ready!


In [3]:
# Get retrieved information from ChromaDB
context = "\n".join(results["documents"][0])

# Create the prompt
prompt = f"""
Answer the question using only the context below.
If the answer is not available, say you don't know.

Context:
{context}

Question:
How many days of annual leave do full-time employees receive?

Answer:
"""

# Tokenize the prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Generate answer
outputs = llm.generate(
    **inputs,
    max_new_tokens=100
)

# Decode answer
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("AI ANSWER:", answer)

NameError: name 'results' is not defined

In [5]:
from sentence_transformers import SentenceTransformer
import chromadb

# Re-initialize client, model, and collection if they are not in memory
try:
    model
except NameError:
    model = SentenceTransformer('all-MiniLM-L6-v2')

try:
    collection
except NameError:
    client = chromadb.Client()
    # Retrieve existing collection
    collection = client.get_collection(name="northstar_handbook")

question = "How many days of annual leave do employees receive?"

question_embedding = model.encode(question).tolist()

results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

print(results["documents"][0])

ModuleNotFoundError: No module named 'chromadb'

In [6]:
!pip install -q sentence-transformers chromadb pypdf transformers accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [7]:
import sentence_transformers
import chromadb
import pypdf
import transformers

print("All libraries are working!")

All libraries are working!


In [8]:
from pypdf import PdfReader

pdf_path = "/content/sample_company_handbook_rag_training.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print("PDF loaded successfully!")
print("Pages:", len(reader.pages))
print("Characters:", len(text))

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_company_handbook_rag_training.pdf'

In [9]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']


In [10]:
import os

filename = "sample_company_handbook_rag_training.pdf"

for folder in ["/", "/root", "/tmp", "/content"]:
    path = os.path.join(folder, filename)

    if os.path.exists(path):
        print("PDF FOUND:", path)

PDF FOUND: /sample_company_handbook_rag_training.pdf


In [11]:
from pypdf import PdfReader

pdf_path = "/sample_company_handbook_rag_training.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print("PDF loaded successfully!")
print("Pages:", len(reader.pages))
print("Characters:", len(text))

PDF loaded successfully!
Pages: 3
Characters: 6648


In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model restored!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model restored!


In [13]:
chunk_size = 800
chunk_overlap = 150

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - chunk_overlap

print("Chunks recreated:", len(chunks))

Chunks recreated: 11


In [14]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="northstar_handbook"
)

chunk_embeddings = model.encode(chunks).tolist()

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings
)

print("Documents stored:", collection.count())

Documents stored: 11


In [15]:
question = "How many days of annual leave do employees receive?"

question_embedding = model.encode(question).tolist()

results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

for i, document in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(document)


Result 1:
ssible if they will be late or absent.
Attendance is recorded through the company’s fictional WorkTrack portal. Employees who cannot access the
portal should email people@northstar.example.
3. Annual leave and other time off
Full-time employees receive 25 days of paid annual leave per calendar year, excluding public holidays observed
in their agreed work location. Part-time leave is calculated proportionally.
Leave requests should be submitted through WorkTrack at least 10 working days before the first requested day.
A manager normally responds within three working days. Approval depends on team coverage and business
needs; employees should not book non-refundable travel before approval.
Employees may take up to five paid sick days per calendar year under this fictional policy. For an abse

Result 2:
ot book non-refundable travel before approval.
Employees may take up to five paid sick days per calendar year under this fictional policy. For an absence
lasting more than two c

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-small"
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

print("FLAN-T5 ready!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 ready!


In [17]:
context = "\n".join(results["documents"][0])

prompt = f"""
Answer the question using only the context.
Be concise and accurate.

Context:
{context}

Question:
{question}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = llm.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("QUESTION:", question)
print("AI ANSWER:", answer)

QUESTION: How many days of annual leave do employees receive?
AI ANSWER: the portal should email people@northstar.example.


In [18]:
# Retrieve only the most relevant chunk

results = collection.query(
    query_embeddings=[model.encode(question).tolist()],
    n_results=1
)

context = results["documents"][0][0]

prompt = f"""
You are a question-answering assistant.

Read the context carefully.
Find the exact answer to the question.
Do not provide unrelated information.

Context: {context}

Question: {question}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = llm.generate(
    **inputs,
    max_new_tokens=80,
    num_beams=4,
    do_sample=False
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("QUESTION:", question)
print("ANSWER:", answer)

QUESTION: How many days of annual leave do employees receive?
ANSWER: 25


In [19]:
def rag_assistant(question):

    # 1. Convert question into embedding
    question_embedding = model.encode(question).tolist()

    # 2. Retrieve relevant document
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=1
    )

    context = results["documents"][0][0]

    # 3. Create prompt
    prompt = f"""
    Answer using only the context below.
    If the answer is unavailable, say you don't know.

    Context: {context}

    Question: {question}

    Answer:
    """

    # 4. Generate answer
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer


print("RAG assistant is ready!")

RAG assistant is ready!


In [20]:
question = "What are the company's standard working hours?"

response = rag_assistant(question)

print("QUESTION:", question)
print("ANSWER:", response)

QUESTION: What are the company's standard working hours?
ANSWER: Monday to Friday, 9:00 a.m. to 5:00 p.m. in the employee’s agreed local time zone, with a one-hour unpaid lunch break


In [21]:
!pip install -q gradio

In [22]:
import gradio as gr

def chat(message, history):

    answer = rag_assistant(message)

    return answer


demo = gr.ChatInterface(
    fn=chat,
    title="Northstar AI Assistant",
    description="Ask me anything about the Northstar employee handbook.",
    textbox=gr.Textbox(
        placeholder="Ask a question about company policies...",
        label="Your question"
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://31295f237e0201fa56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
import os

project_folder = "/content/drive/MyDrive/Northstar AI - RAG Assistant"

os.makedirs(project_folder, exist_ok=True)

print("Project folder created successfully!")
print(project_folder)

Project folder created successfully!
/content/drive/MyDrive/Northstar AI - RAG Assistant


In [25]:
import shutil

shutil.copy(
    "/sample_company_handbook_rag_training.pdf",
    project_folder
)

print("PDF backed up successfully!")

PDF backed up successfully!


In [26]:
import os

# Create project directories
folders = [
    "screenshots",
    "sample_data"
]

for folder in folders:
    os.makedirs(os.path.join(project_folder, folder), exist_ok=True)

print("Project structure created!")

Project structure created!


In [27]:
requirements = """
sentence-transformers
chromadb
pypdf
transformers
accelerate
gradio
torch
"""

with open(os.path.join(project_folder, "requirements.txt"), "w") as f:
    f.write(requirements.strip())

print("requirements.txt created!")

requirements.txt created!


In [28]:
readme = """
# Northstar AI – RAG Assistant

An AI-powered document question-answering assistant built with
Retrieval-Augmented Generation (RAG).

## Overview

Northstar AI allows users to ask questions about an employee handbook
and receive answers grounded in the document's content.

## Technology Stack

- Python
- Sentence Transformers
- ChromaDB
- Hugging Face Transformers
- FLAN-T5-small
- PyPDF
- Gradio
- Google Colab

## How It Works

1. Extract text from PDF documents.
2. Split documents into overlapping text chunks.
3. Generate semantic embeddings using Sentence Transformers.
4. Store embeddings in ChromaDB.
5. Retrieve relevant information through semantic search.
6. Generate answers using FLAN-T5-small.
7. Display responses through a Gradio chatbot.

## Features

- PDF document ingestion
- Semantic search
- Vector database storage
- Context-based answer generation
- Interactive chatbot interface
- No paid OpenAI API key required

## Project Structure

- notebooks/ - Project notebook
- sample_data/ - Sample documents
- screenshots/ - Application screenshots
- requirements.txt - Dependencies

## Future Improvements

- Multiple document uploads
- Source citations
- Improved retrieval evaluation
- Persistent vector storage
- Permanent deployment

## Author

Esther Usman

AI Automation Engineer | AI Agent Developer
"""

with open(os.path.join(project_folder, "README.md"), "w") as f:
    f.write(readme)

print("README.md created successfully!")

README.md created successfully!


In [29]:
for root, dirs, files in os.walk(project_folder):
    print(root, files)

/content/drive/MyDrive/Northstar AI - RAG Assistant ['sample_company_handbook_rag_training.pdf', 'requirements.txt', 'README.md']
/content/drive/MyDrive/Northstar AI - RAG Assistant/screenshots []
/content/drive/MyDrive/Northstar AI - RAG Assistant/sample_data []


In [30]:
import os
import shutil

notebook_folder = os.path.join(project_folder, "notebooks")
os.makedirs(notebook_folder, exist_ok=True)

print("Notebook folder created!")

Notebook folder created!
